<a href="https://colab.research.google.com/github/pinkyproject25-cmd/crop-diseasee-ai/blob/main/notebooks/train_classifier_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Disease AI — parallel classifier training

This notebook is exclusively for `pinkyproject25-cmd/crop-diseasee-ai`. It creates resumable **candidate** artifacts in a new Google Drive folder. It never reads from or writes to the original project notebook or deployments. Candidate output remains `production_approved: false`; field-image, realistic OOD, severity, and agricultural-review gates remain mandatory.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi
# Keep Colab's tested GPU build of PyTorch; install only the extra tools.
!pip install -q 'datasets==3.6.0' 'huggingface-hub>=0.24,<1.0' 'onnx==1.18.0' 'onnxruntime==1.21.1' 'scikit-learn==1.7.2' 'matplotlib==3.10.6' 'tqdm==4.67.1'

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CropDiseaseAIParallel')
OUTPUT_DIR = DRIVE_ROOT / 'classifier_candidate_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Persistent candidate output: {OUTPUT_DIR}')

Mounted at /content/drive
Persistent candidate output: /content/drive/MyDrive/CropDiseaseAIParallel/classifier_candidate_v1


In [ ]:
import subprocess

REPOSITORY = 'https://github.com/pinkyproject25-cmd/crop-diseasee-ai.git'
FORBIDDEN_REPOSITORY = 'Ashish7386/crop-disease-ai'
WORKTREE = Path('/content/crop-diseasee-ai-parallel')

if WORKTREE.exists():
    origin = subprocess.check_output(['git', '-C', str(WORKTREE), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != REPOSITORY or FORBIDDEN_REPOSITORY in origin:
        raise RuntimeError(f'Refusing unexpected Git origin: {origin}')
    subprocess.run(['git', '-C', str(WORKTREE), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKTREE), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(WORKTREE), 'merge', '--ff-only', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', '--single-branch', REPOSITORY, str(WORKTREE)], check=True)

origin = subprocess.check_output(['git', '-C', str(WORKTREE), 'remote', 'get-url', 'origin'], text=True).strip()
commit = subprocess.check_output(['git', '-C', str(WORKTREE), 'rev-parse', 'HEAD'], text=True).strip()
assert origin == REPOSITORY and FORBIDDEN_REPOSITORY not in origin
print({'origin': origin, 'commit': commit})

In [ ]:
command = [
    'python', str(WORKTREE / 'training/train_classifier.py'),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '12',
    '--warmup-epochs', '2',
    '--batch-size', '64',
    '--num-workers', '2',
    '--val-fold', '0',
    '--calibration-fold', '1',
    '--test-fold', '2',
    '--split-folds', '10',
]
print('Starting or resuming:', ' '.join(command))
subprocess.run(command, cwd=WORKTREE, check=True)

In [ ]:
import json

metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
manifest = json.loads((OUTPUT_DIR / 'candidate_manifest.json').read_text(encoding='utf-8'))
assert metrics['production_approved'] is False
assert manifest['production_approved'] is False
summary = {
    'model_version': metrics['model_version'],
    'dataset_revision': metrics['dataset']['revision'],
    'test_accuracy': metrics['test']['accuracy'],
    'test_macro_f1': metrics['test']['macro_f1'],
    'test_ece': metrics['test']['ece'],
    'threshold': metrics['acceptance_threshold'],
    'test_selective_metrics': metrics['independent_test_selective_metrics'],
    'onnx_verification': metrics['onnx_verification'],
    'approval_blockers': metrics['approval_blockers'],
}
print(json.dumps(summary, indent=2))
print('Review:', OUTPUT_DIR / 'test_confusion_matrix.png')
print('Per-class CSV:', OUTPUT_DIR / 'test_per_class_metrics.csv')

{
  "model_version": "plantvillage-mnv3-20261006T022813Z",
  "dataset_revision": "9e97599868962bd0079b8db4b7f1efa9185fa1e7",
  "test_accuracy": 0.9933456561922366,
  "test_macro_f1": 0.9915131174459015,
  "test_ece": 0.0025011349737974757,
  "threshold": {
    "status": "provisional_ood_sanity_check",
    "threshold": 0.99,
    "validation_coverage": 0.93980117820324,
    "accepted_precision": 0.9992164544564153,
    "cifar100_false_acceptance": 0.0071
  },
  "test_selective_metrics": {
    "threshold": 0.99,
    "coverage": 0.9460258780036969,
    "accepted_count": 5118,
    "rejected_count": 292,
    "accepted_accuracy": 0.9988276670574443
  },
  "onnx_verification": {
    "examples": 8,
    "top1_agreement": 8,
    "max_absolute_logit_error": 3.814697265625e-05
  },
  "approval_blockers": [
    "Independent field-photo evaluation has not passed.",
    "Unsupported crop and non-leaf rejection has not been validated beyond the CIFAR-100 sanity check.",
    "Disease-area segmentation a

In [ ]:
import shutil

archive = shutil.make_archive(str(DRIVE_ROOT / 'crop-disease-classifier-candidate-v1'), 'zip', OUTPUT_DIR)
print(f'Candidate archive saved persistently at {archive}')
print('Do not copy it into backend/models until every approval blocker is resolved.')

Candidate archive saved persistently at /content/drive/MyDrive/CropDiseaseAIParallel/crop-disease-classifier-candidate-v1.zip
Do not copy it into backend/models until every approval blocker is resolved.
